# colab-drive
Hệ thống tự động hóa tải và đồng bộ tệp tin đa nguồn vào Google Drive trên nền tảng Google Colab.

Chức năng chính:
- Hỗ trợ đa giao thức: HTTP/HTTPS direct link, HLS (.m3u8), DASH (.mpd), BitTorrent/Magnet, và hơn 1.800 nền tảng trực tuyến qua yt-dlp.
- Hỗ trợ mạng máy chủ ủy danh (Proxy HTTP/HTTPS/SOCKS5) và tùy biến tiêu đề yêu cầu (User-Agent, Referer).
- Tối ưu băng thông tải xuống bằng tiến trình đa luồng aria2c.
- Tự động đồng bộ hóa dữ liệu trực tiếp vào Google Drive cá nhân.

Hướng dẫn vận hành: Thực thi tuần tự các bước từ Bước 1 đến Bước 3.


In [ ]:
#@title Bước 1: Thiết lập môi trường và cài đặt phụ thuộc
#@markdown Khởi tạo môi trường thực thi, cài đặt các gói tiện ích gồm yt-dlp, ffmpeg và aria2c.

import os
import sys
import time

def log(tag, msg, level="INFO"):
    colors = {
        "SYSTEM": "\033[36m",
        "SUCCESS": "\033[32m",
        "INFO": "\033[34m",
        "WARN": "\033[33m",
        "ERROR": "\033[31m",
        "DRIVE": "\033[35m",
        "FETCH": "\033[94m",
        "TRANSFER": "\033[33m"
    }
    reset = "\033[0m"
    dim = "\033[90m"
    ts = time.strftime("%H:%M:%S")
    tag_color = colors.get(level, "\033[37m")
    print(f"{dim}{ts}{reset} {tag_color}[{tag:<8}]{reset} {msg}")

log("SYSTEM", "Bắt đầu khởi tạo môi trường hệ thống...", "SYSTEM")
log("SYSTEM", "Cài đặt và cập nhật gói yt-dlp...", "INFO")
os.system("pip install --upgrade --no-cache-dir -q yt-dlp")

log("SYSTEM", "Cài đặt aria2c và ffmpeg qua trình quản lý gói APT...", "INFO")
os.system("apt-get update -y -q && apt-get install -y -q ffmpeg aria2")

log("SUCCESS", "Thiết lập môi trường và phụ thuộc hoàn tất.", "SUCCESS")



In [ ]:
#@title Bước 2: Xác thực và kết nối Google Drive
#@markdown Liên kết phân vùng lưu trữ Google Drive để đồng bộ dữ liệu tải xuống.

import os
import time

def log(tag, msg, level="INFO"):
    colors = {
        "SYSTEM": "\033[36m",
        "SUCCESS": "\033[32m",
        "INFO": "\033[34m",
        "WARN": "\033[33m",
        "ERROR": "\033[31m",
        "DRIVE": "\033[35m",
        "FETCH": "\033[94m",
        "TRANSFER": "\033[33m"
    }
    reset = "\033[0m"
    dim = "\033[90m"
    ts = time.strftime("%H:%M:%S")
    tag_color = colors.get(level, "\033[37m")
    print(f"{dim}{ts}{reset} {tag_color}[{tag:<8}]{reset} {msg}")

log("DRIVE", "Yêu cầu quyền truy cập phân vùng Google Drive...", "DRIVE")
try:
    from google.colab import drive
    drive.mount('/content/drive')
    if os.path.exists('/content/drive/MyDrive'):
        log("SUCCESS", "Kết nối Google Drive thành công.", "SUCCESS")
        log("DRIVE", "Điểm gắn lưu trữ: /content/drive/MyDrive/", "INFO")
    else:
        log("WARN", "Đã gắn kết phân vùng nhưng chưa nhận diện được thư mục MyDrive.", "WARN")
except Exception as e:
    log("ERROR", f"Kết nối Google Drive thất bại: {e}", "ERROR")



In [ ]:
#@title Bước 3: Tải xuống và đồng bộ vào Google Drive
#@markdown Nhập liên kết cần tải về (hỗ trợ Torrent/Magnet, Video Stream, Direct link, và mọi nền tảng được yt-dlp hỗ trợ).

URL_INPUT = "" #@param {type:"string"}
DRIVE_FOLDER = "/content/drive/MyDrive/colab-drive" #@param {type:"string"}
CUSTOM_FILENAME = "" #@param {type:"string"}
PROXY_LIST = "" #@param {type:"string"}
USE_ARIA2 = True #@param {type:"boolean"}
CUSTOM_REFERER = "" #@param {type:"string"}

import os
import re
import glob
import shutil
import random
import time
import urllib.parse
import yt_dlp

def log(tag, msg, level="INFO"):
    colors = {
        "SYSTEM": "\033[36m",
        "SUCCESS": "\033[32m",
        "INFO": "\033[34m",
        "WARN": "\033[33m",
        "ERROR": "\033[31m",
        "DRIVE": "\033[35m",
        "FETCH": "\033[94m",
        "TRANSFER": "\033[33m"
    }
    reset = "\033[0m"
    dim = "\033[90m"
    ts = time.strftime("%H:%M:%S")
    tag_color = colors.get(level, "\033[37m")
    print(f"{dim}{ts}{reset} {tag_color}[{tag:<8}]{reset} {msg}")

def parse_proxies(proxy_raw):
    proxies = []
    if not proxy_raw or not proxy_raw.strip():
        return proxies
    raw_clean = proxy_raw.replace(',', '\n').replace(';', '\n').replace(' ', '\n')
    for line in raw_clean.split('\n'):
        line = line.strip()
        if not line:
            continue
        parts = line.split(':')
        if len(parts) == 4:
            ip, port, user, pwd = parts
            proxies.append(f"http://{user}:{pwd}@{ip}:{port}")
        elif len(parts) == 2:
            ip, port = parts
            proxies.append(f"http://{ip}:{port}")
        elif line.startswith(('http://', 'https://', 'socks5://', 'socks4://')):
            proxies.append(line)
    return proxies

def mask_proxy(p_str):
    return re.sub(r'//.*?:.*?@', '//***:***@', p_str)

def get_unique_filename(dest_dir, filename):
    base, ext = os.path.splitext(filename)
    counter = 1
    new_filename = filename
    while os.path.exists(os.path.join(dest_dir, new_filename)):
        new_filename = f"{base}_{counter}{ext}"
        counter += 1
    return new_filename

if not URL_INPUT.strip():
    log("ERROR", "Chưa có thông tin liên kết đầu vào.", "ERROR")
    raise ValueError("Vui lòng nhập liên kết hợp lệ tại ô URL_INPUT.")

raw_urls = URL_INPUT.replace(',', '\n').replace(';', '\n')
urls = [u.strip() for u in raw_urls.split('\n') if u.strip()]

proxies = parse_proxies(PROXY_LIST)
temp_workspace = "/content/temp_download"

if os.path.exists(temp_workspace):
    shutil.rmtree(temp_workspace, ignore_errors=True)
os.makedirs(temp_workspace, exist_ok=True)
os.makedirs(DRIVE_FOLDER, exist_ok=True)

log("SYSTEM", f"Khởi chạy tiến trình xử lý cho {len(urls)} liên kết.", "SYSTEM")
if proxies:
    log("SYSTEM", f"Đã nạp {len(proxies)} địa chỉ proxy.", "INFO")

success_count = 0
failed_count = 0

for index, target_url in enumerate(urls, start=1):
    log("FETCH", f"[{index}/{len(urls)}] Đang xử lý: {target_url}", "FETCH")
    
    current_proxy = random.choice(proxies) if proxies else None
    if current_proxy:
        log("SYSTEM", f"Định tuyến qua proxy: {mask_proxy(current_proxy)}", "INFO")
        
    sub_task_dir = os.path.join(temp_workspace, f"task_{index:03d}")
    os.makedirs(sub_task_dir, exist_ok=True)
    
    is_torrent = target_url.startswith('magnet:?') or target_url.lower().endswith('.torrent') or '.torrent?' in target_url.lower()
    
    download_ok = False
    
    if is_torrent:
        log("FETCH", "Phát hiện định dạng BitTorrent/Magnet. Khởi chạy aria2c...", "FETCH")
        trackers = [
            "udp://tracker.opentrackr.org:1337/announce",
            "udp://open.tracker.cl:1337/announce",
            "udp://9.rarbg.com:2810/announce",
            "udp://tracker.openbittorrent.com:6969/announce",
            "http://tracker.openbittorrent.com:80/announce"
        ]
        tracker_arg = f"--bt-tracker={','.join(trackers)}"
        aria2_cmd = (
            f'aria2c --seed-time=0 --summary-interval=5 --file-allocation=none '
            f'--enable-dht=true --dht-listen-port=6881 --bt-enable-lpd=true '
            f'{tracker_arg} --dir="{sub_task_dir}" "{target_url}"'
        )
        if current_proxy:
            aria2_cmd += f' --all-proxy="{current_proxy}"'
            
        ret = os.system(aria2_cmd)
        if ret == 0:
            download_ok = True
        else:
            log("ERROR", f"Tiến trình aria2c báo lỗi với mã thoát: {ret}", "ERROR")
    else:
        headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36',
            'Accept': '*/*',
            'Accept-Language': 'en-US,en;q=0.9,vi;q=0.8'
        }
        if CUSTOM_REFERER.strip():
            headers['Referer'] = CUSTOM_REFERER.strip()
            headers['Origin'] = urllib.parse.urlsplit(CUSTOM_REFERER.strip()).scheme + "://" + urllib.parse.urlsplit(CUSTOM_REFERER.strip()).netloc
        elif 'seenow' in target_url.lower() or 'vtv' in target_url.lower():
            headers['Referer'] = 'https://vtvprime.vn/'
            headers['Origin'] = 'https://vtvprime.vn'

        out_template = os.path.join(sub_task_dir, '%(title)s.%(ext)s')
        if CUSTOM_FILENAME.strip():
            custom_name = CUSTOM_FILENAME.strip()
            if len(urls) > 1:
                name_part, ext_part = os.path.splitext(custom_name)
                custom_name = f"{name_part}_{index}{ext_part}"
            out_template = os.path.join(sub_task_dir, custom_name)

        ydl_opts = {
            'outtmpl': out_template,
            'format': 'bestvideo*+bestaudio/best',
            'merge_output_format': 'mp4',
            'quiet': False,
            'no_warnings': True,
            'retries': 10,
            'fragment_retries': 10,
            'concurrent_fragment_downloads': 8,
            'http_headers': headers
        }
        if current_proxy:
            ydl_opts['proxy'] = current_proxy

        if USE_ARIA2:
            ydl_opts['external_downloader'] = 'aria2c'
            aria2_opts = ['-x16', '-s16', '-k1M', '--file-allocation=none']
            if current_proxy:
                aria2_opts.append(f'--all-proxy={current_proxy}')
            if 'Referer' in headers:
                aria2_opts.append(f'--header=Referer: {headers["Referer"]}')
            if 'Origin' in headers:
                aria2_opts.append(f'--header=Origin: {headers["Origin"]}')
            ydl_opts['external_downloader_args'] = aria2_opts

        ext_dl = 'aria2c' if USE_ARIA2 else 'native'
        log("FETCH", f"Khởi chạy yt-dlp (công cụ tải ngoài: {ext_dl})...", "INFO")
        try:
            with yt_dlp.YoutubeDL(ydl_opts) as ydl:
                ydl.download([target_url])
            download_ok = True
        except Exception as err:
            log("WARN", f"Phát sinh lỗi trong quá trình tải ({err}). Tiến hành chế độ dự phòng format 'best'...", "WARN")
            ydl_opts['format'] = 'best'
            try:
                with yt_dlp.YoutubeDL(ydl_opts) as ydl:
                    ydl.download([target_url])
                download_ok = True
            except Exception as err2:
                log("WARN", f"Chế độ dự phòng yt-dlp thất bại: {err2}. Thử nghiệm tải trực tiếp bằng aria2c...", "WARN")
                direct_cmd = f'aria2c -x16 -s16 -k1M --file-allocation=none -d "{sub_task_dir}" "{target_url}"'
                if current_proxy:
                    direct_cmd += f' --all-proxy="{current_proxy}"'
                if os.system(direct_cmd) == 0:
                    download_ok = True
                else:
                    log("ERROR", f"Không thể tải xuống tệp từ nguồn: {target_url}", "ERROR")

    # Scanning output files
    output_items = []
    for root, dirs, files in os.walk(sub_task_dir):
        for f in files:
            if not f.endswith(('.part', '.ytdl', '.aria2')):
                output_items.append(os.path.join(root, f))

    if download_ok and output_items:
        log("TRANSFER", f"Chuyển giao {len(output_items)} tệp tin vào Google Drive: {DRIVE_FOLDER}", "TRANSFER")
        for fpath in output_items:
            fname = os.path.basename(fpath)
            target_fname = get_unique_filename(DRIVE_FOLDER, fname)
            dest_path = os.path.join(DRIVE_FOLDER, target_fname)
            shutil.move(fpath, dest_path)
            log("SUCCESS", f"Lưu trữ thành công: {dest_path}", "SUCCESS")
        success_count += 1
    else:
        log("ERROR", f"Thất bại khi xử lý tiến trình [{index}/{len(urls)}]", "ERROR")
        failed_count += 1

try:
    shutil.rmtree(temp_workspace, ignore_errors=True)
except:
    pass

print("\n" + "="*60)
log("SYSTEM", f"Tổng kết: {success_count} thành công, {failed_count} thất bại.", "SYSTEM")
log("DRIVE", f"Thư mục lưu trữ trên Google Drive: {DRIVE_FOLDER}", "DRIVE")
print("="*60)

